# Fine-Tuning GPT-2 for Rap Lyrics Generation

## Project Goal
The goal of this project is to fine-tune a pre-trained GPT-2 language model to generate rap lyrics in the style of the artist Drake, using Hugging Face's `transformers` and `datasets` libraries.

In [47]:
# 1. Install Necessary Libraries
# Using -q for quieter output
!pip install transformers datasets torch accelerate evaluate pandas -q
print("Libraries installed.")

Libraries installed.


In [48]:
# 2. Import Libraries
import pandas as pd
import re
import os
import torch
from datasets import load_dataset, DatasetDict, Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)
# For potentially checking GPU availability later if not using pipeline's auto-detect
# from accelerate import Accelerator

print("Libraries imported.")

Libraries imported.


## Data Acquisition and Cleaning

The dataset used for fine-tuning consists of Drake song lyrics sourced from Kaggle: [Drake Lyrics Dataset](https://www.kaggle.com/datasets/juicobowley/drake-lyrics).

The dataset is provided as a CSV file (`drake_data.csv`). We first load this data using pandas and extract the raw lyrics text.

The raw lyrics contain metadata tags (like `[Verse]`, `[Chorus]`, etc.) and potential inconsistencies in whitespace and newlines. We apply a cleaning function using regular expressions to:
* Remove all bracketed tags.
* Standardize whitespace and newlines.
* Remove empty lines.

The cleaned lyrics are saved to `cleaned-drake-lyrics.txt` for subsequent processing.

In [49]:
# 3. Load Raw Data from CSV

# Configuration
csv_file_path = 'drake_data.csv'
output_raw_lyrics_file = 'raw-drake-lyrics.txt' # For inspection or alternative loading

print(f"Attempting to load lyrics from {csv_file_path}...")

# Ensure the file exists before trying to read
if not os.path.exists(csv_file_path):
    print(f"ERROR: File not found at {csv_file_path}. Please upload it.")
    all_lyrics_list = None # Set flag indicating failure
else:
    try:
        # Load CSV using pandas
        df = pd.read_csv(csv_file_path)
        print(f"Successfully loaded {csv_file_path} ({len(df)} songs).")

        # Check if 'lyrics' column exists
        if 'lyrics' in df.columns:
            # Extract the 'lyrics' column
            # Fill missing lyrics (NaN) with empty strings and ensure all are strings
            all_lyrics_list = df['lyrics'].fillna('').astype(str).tolist()
            print(f"Successfully extracted lyrics for {len(all_lyrics_list)} songs.")

            # Save the raw extracted lyrics (joined by double newline) to a file
            raw_lyrics_content = '\n\n'.join(all_lyrics_list) # Use \n\n to separate songs
            with open(output_raw_lyrics_file, 'w', encoding='utf-8') as f:
                f.write(raw_lyrics_content)
            print(f"Saved raw lyrics to {output_raw_lyrics_file}")

            # Display head of dataframe and example lyrics
            print("\nDataFrame Head:")
            print(df.head())
            print(f"\nExample: First 200 characters of the first song's raw lyrics:")
            if all_lyrics_list:
                print(all_lyrics_list[0][:200])

        else:
            print(f"ERROR: 'lyrics' column not found in {csv_file_path}.")
            all_lyrics_list = None

    except Exception as e:
        print(f"An error occurred loading or processing the CSV: {e}")
        all_lyrics_list = None

Attempting to load lyrics from drake_data.csv...
Successfully loaded drake_data.csv (290 songs).
Successfully extracted lyrics for 290 songs.
Saved raw lyrics to raw-drake-lyrics.txt

DataFrame Head:
                 album                           lyrics_title  \
0  Certified Lover Boy            Certified Lover Boy* Lyrics   
1  Certified Lover Boy  Like I’m Supposed To/Do Things Lyrics   
2  Certified Lover Boy                      Not Around Lyrics   
3  Certified Lover Boy    In the Cut (Ft. Roddy Ricch) Lyrics   
4  Certified Lover Boy  Zodiac Sign (Ft. Jessie Reyez) Lyrics   

                                          lyrics_url  \
0  https://genius.com/Drake-certified-lover-boy-l...   
1  https://genius.com/Drake-like-im-supposed-to-d...   
2         https://genius.com/Drake-not-around-lyrics   
3         https://genius.com/Drake-in-the-cut-lyrics   
4        https://genius.com/Drake-zodiac-sign-lyrics   

                                              lyrics track_views  
0  [V

In [50]:
# 4. Clean Raw Lyrics Data

# Configuration
input_raw_file = 'raw-drake-lyrics.txt'
output_cleaned_file = 'cleaned-drake-lyrics.txt'

# Define the cleaning function
def clean_lyrics(raw_text):
    """Cleans raw lyric text by removing tags and standardizing whitespace."""
    # Remove bracketed text (e.g., [Intro], [Chorus], etc.)
    cleaned_text = re.sub(r'\[[^\]]*?\]', '', raw_text)
    # Fix extra whitespace (multiple spaces/tabs to single space)
    cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)
    # Fix extra newlines (multiple newlines to single newline)
    cleaned_text = re.sub(r'\n+', '\n', cleaned_text)
    # Split into lines, strip leading/trailing whitespace from each
    lines = cleaned_text.split('\n')
    cleaned_lines = [line.strip() for line in lines]
    # Filter out empty lines
    cleaned_lines = [line for line in cleaned_lines if line]
    # Join back with single newlines
    final_text = '\n'.join(cleaned_lines)
    return final_text

# --- Load Raw Lyrics from file ---
print(f"Loading raw lyrics from {input_raw_file} for cleaning...")
if not os.path.exists(input_raw_file):
    print(f"ERROR: Raw lyrics file not found: {input_raw_file}")
else:
    try:
        with open(input_raw_file, 'r', encoding='utf-8') as f:
            raw_drake_lyrics = f.read()
        print(f"Raw lyrics loaded.")

        # --- Clean Lyrics ---
        print("Cleaning lyrics...")
        cleaned_drake_lyrics = clean_lyrics(raw_drake_lyrics)
        print("Cleaning complete.")

        # --- Save Cleaned Lyrics ---
        with open(output_cleaned_file, 'w', encoding='utf-8') as f:
            f.write(cleaned_drake_lyrics)
        print(f"Cleaned lyrics saved to {output_cleaned_file}")

        # --- Show Snippet ---
        print(f"\nExample: First 300 characters of the cleaned lyrics:")
        print(cleaned_drake_lyrics[:300])

    except Exception as e:
        print(f"An error occurred during cleaning: {e}")

Loading raw lyrics from raw-drake-lyrics.txt for cleaning...
Raw lyrics loaded.
Cleaning lyrics...
Cleaning complete.
Cleaned lyrics saved to cleaned-drake-lyrics.txt

Example: First 300 characters of the cleaned lyrics:
Put my feelings on ice
Always been a gem
Certified lover boy, somehow still heartless
Heart is only gettin' colder
Hands are tied
Someone's in my ear from the other side
Tellin' me that I should pay you no mind
Wanted you to not be with me all night
Wanted you to not stay with me all night
I know, y


## Exploratory Data Analysis (EDA)

Before proceeding with model training, we perform a basic Exploratory Data Analysis (EDA) on the cleaned dataset (`cleaned-drake-lyrics.txt`). This helps us understand the size and scale of our training data. We count the total number of non-empty lines and the total number of words.

In [51]:
# 5. Perform Exploratory Data Analysis (EDA)

# Configuration
cleaned_file = 'cleaned-drake-lyrics.txt'

print(f"Loading cleaned lyrics from {cleaned_file} for EDA...")
if not os.path.exists(cleaned_file):
    print(f"ERROR: Cleaned lyrics file not found: {cleaned_file}")
else:
    try:
        # Read the entire cleaned text file
        with open(cleaned_file, 'r', encoding='utf-8') as f:
            drake_text = f.read()
        print("File loaded.")

        # --- Calculate Stats ---
        # Split into lines and filter out any empty lines
        lines = [line for line in drake_text.split('\n') if line.strip()]
        num_lines = len(lines)

        # Split into words based on whitespace
        words = drake_text.split()
        num_words = len(words)

        # --- Print Results ---
        print("\n--- Basic EDA Results ---")
        print(f"Number of non-empty lines: {num_lines}")
        print(f"Total number of words: {num_words}")

        # Show a few example lines from the cleaned data
        print("\nFirst 10 lines of cleaned data:")
        for i, line in enumerate(lines[:10]):
             print(f"{i+1}: {line}")

    except Exception as e:
        print(f"An error occurred during EDA: {e}")

Loading cleaned lyrics from cleaned-drake-lyrics.txt for EDA...
File loaded.

--- Basic EDA Results ---
Number of non-empty lines: 17580
Total number of words: 147131

First 10 lines of cleaned data:
1: Put my feelings on ice
2: Always been a gem
3: Certified lover boy, somehow still heartless
4: Heart is only gettin' colder
5: Hands are tied
6: Someone's in my ear from the other side
7: Tellin' me that I should pay you no mind
8: Wanted you to not be with me all night
9: Wanted you to not stay with me all night
10: I know, you know, who that person is to me


## Model and Tokenizer Setup

For this project, we use the standard pre-trained **GPT-2** model (`gpt2`) from the Hugging Face model hub as our base language model. We also load its corresponding tokenizer.

The tokenizer converts the text lyrics into numerical IDs that the model can understand. We explicitly set the tokenizer's `pad_token` to its `eos_token` (end-of-sentence token), which is a common practice for causal language models like GPT-2.

In [52]:
# 6. Load Base Model and Tokenizer

# Configuration
model_checkpoint = "gpt2" # Define the base model

print(f"Loading tokenizer for '{model_checkpoint}'...")
try:
    # Load the tokenizer associated with the chosen model checkpoint
    # use_fast=True usually provides a faster implementation if available
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

    # Set the padding token. For GPT-2, we use the end-of-sentence token as the padding token.
    tokenizer.pad_token = tokenizer.eos_token
    print("Tokenizer loaded successfully.")
    print(f"Pad token set to: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

except Exception as e:
    print(f"Error loading tokenizer: {e}")
    tokenizer = None

print(f"\nLoading base model for '{model_checkpoint}'...")
if tokenizer: # Only proceed if tokenizer loaded
    try:
        # Load the pre-trained model for causal language modeling
        model = AutoModelForCausalLM.from_pretrained(model_checkpoint)
        print("Base GPT-2 model loaded successfully.")

        # Ensure the model's pad_token_id is also set (important for generation)
        # Although Trainer handles attention mask, setting this explicitly can be good practice
        model.config.pad_token_id = tokenizer.pad_token_id
        print(f"Model config pad_token_id set to: {model.config.pad_token_id}")

    except Exception as e:
        print(f"Error loading model: {e}")
        model = None
else:
    print("Skipping model loading due to tokenizer error.")
    model = None

# Basic check
if model and tokenizer:
    print("\nBase model and tokenizer are ready.")
else:
    print("\nFailed to load base model and/or tokenizer.")

Loading tokenizer for 'gpt2'...
Tokenizer loaded successfully.
Pad token set to: <|endoftext|> (ID: 50256)

Loading base model for 'gpt2'...
Base GPT-2 model loaded successfully.
Model config pad_token_id set to: 50256

Base model and tokenizer are ready.


## Data Preparation for Training

Before training, the cleaned text data needs to be processed into a format suitable for the language model:

1.  **Load Data:** The `cleaned-drake-lyrics.txt` file is loaded into a Hugging Face `Dataset` object.
2.  **Tokenization:** The loaded text is tokenized using the pre-loaded GPT-2 tokenizer. This converts lines of text into sequences of numerical IDs.
3.  **Blocking:** Causal language models are typically trained on fixed-length blocks of text. The tokenized sequences are concatenated and then split into chunks of a specified `block_size` (e.g., 128 tokens). Texts shorter than the block size are effectively dropped after concatenation unless padding is explicitly handled (which we are not doing here for simplicity, as the concatenation minimizes data loss). For Causal LM, the `labels` (what the model tries to predict) are the same as the `input_ids`, shifted internally by the model during training.

The final processed dataset, containing the input blocks, is saved to disk for potential reuse.

In [53]:
# 7. Tokenize and Block Data for Language Modeling

# Configuration
cleaned_file_path = 'cleaned-drake-lyrics.txt'
block_size = 128 # The sequence length for model training blocks
processed_dataset_path = "./drake_lm_dataset" # Directory to save processed data

# Make sure tokenizer is loaded from step 6
if 'tokenizer' not in globals() or tokenizer is None:
    print("ERROR: Tokenizer not loaded. Please run section 4 first.")
else:
    # --- 1. Load Cleaned Data into Hugging Face Dataset ---
    print(f"\nLoading cleaned data from {cleaned_file_path} into Dataset object...")
    if not os.path.exists(cleaned_file_path):
         print(f"ERROR: Cleaned file not found at {cleaned_file_path}.")
         raw_datasets = None
    else:
         try:
             # Load text file directly into a Dataset with a 'train' split
             raw_datasets = load_dataset('text', data_files={'train': cleaned_file_path})
             print("Dataset loaded successfully:")
             print(raw_datasets)
         except Exception as e:
             print(f"Error loading dataset from text file: {e}")
             raw_datasets = None

    if raw_datasets:
        # --- 2. Define Tokenization Function ---
        def tokenize_function(examples):
            # Uses the pre-loaded tokenizer
            return tokenizer(examples["text"])

        # --- 3. Tokenize the Dataset ---
        print("\nTokenizing the dataset...")
        try:
            # Process in batches for efficiency, use multiple processes
            # Remove original 'text' column as it's no longer needed
            tokenized_datasets = raw_datasets.map(
                tokenize_function,
                batched=True,
                num_proc=os.cpu_count(), # Use available CPU cores
                remove_columns=["text"]
            )
            print("Tokenization complete:")
            print(tokenized_datasets)
            # print(f"Example tokenized input_ids: {tokenized_datasets['train'][0]['input_ids'][:50]}...") # Optional: view example IDs
        except Exception as e:
             print(f"Error during tokenization: {e}")
             tokenized_datasets = None

        if tokenized_datasets:
            # --- 4. Define Blocking Function ---
            def group_texts(examples):
                # Concatenate all texts from the batch
                concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
                total_length = len(concatenated_examples[list(examples.keys())[0]])
                # Drop the remainder tokens if total_length is not a multiple of block_size.
                # Adjust if padding is needed, but dropping is common for Causal LM.
                if total_length >= block_size:
                    total_length = (total_length // block_size) * block_size
                # Split into chunks of block_size
                result = {
                    k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
                    for k, t in concatenated_examples.items()
                }
                # Create labels (for Causal LM, labels are input_ids shifted during training)
                result["labels"] = result["input_ids"].copy()
                return result

            # --- 5. Apply Blocking ---
            print("\nGrouping tokenized data into blocks...")
            try:
                lm_datasets = tokenized_datasets.map(
                    group_texts,
                    batched=True,
                    # batch_size=1000, # Process 1000 lines at a time (adjust based on RAM)
                    num_proc=os.cpu_count(), # Use available CPU cores
                )
                print("Blocking complete:")
                print(lm_datasets)

                # --- Final Check & Save ---
                if 'train' in lm_datasets and len(lm_datasets['train']) > 0:
                     num_blocks = len(lm_datasets['train'])
                     print(f"\nSuccessfully created {num_blocks} blocks of size {block_size} for training.")
                     # Save the final processed dataset
                     lm_datasets.save_to_disk(processed_dataset_path)
                     print(f"Processed dataset saved to {processed_dataset_path}")
                else:
                     print("\nERROR: Blocking resulted in an empty or invalid dataset.")
                     lm_datasets = None # Flag error

            except Exception as e:
                print(f"Error during blocking: {e}")
                lm_datasets = None # Flag error
        else:
             print("Skipping blocking due to tokenization error.")
             lm_datasets = None
    else:
        print("Skipping tokenization/blocking due to dataset loading error.")
        lm_datasets = None


Loading cleaned data from cleaned-drake-lyrics.txt into Dataset object...


Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 17580
    })
})

Tokenizing the dataset...


Map (num_proc=44):   0%|          | 0/17580 [00:00<?, ? examples/s]

Tokenization complete:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 17580
    })
})

Grouping tokenized data into blocks...


Map (num_proc=44):   0%|          | 0/17580 [00:00<?, ? examples/s]

Blocking complete:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1492
    })
})

Successfully created 1492 blocks of size 128 for training.


Saving the dataset (0/1 shards):   0%|          | 0/1492 [00:00<?, ? examples/s]

Processed dataset saved to ./drake_lm_dataset


## Model Fine-Tuning

We use the Hugging Face `Trainer` class to handle the fine-tuning process. This simplifies training loops, distributed training (on TPUs/GPUs), and checkpoint management.

We define `TrainingArguments` to configure the training process:
* `output_dir`: Where to save model checkpoints and logs.
* `num_train_epochs`: How many times to iterate over the entire dataset (we use 15 epochs based on experimentation).
* `per_device_train_batch_size`: Number of examples per batch for each TPU core (set to 4).
* `learning_rate`: The rate at which the model adjusts its weights (kept at 5e-5).
* `save_strategy`: Save a checkpoint after each epoch.
* `logging_steps`: How often to log training loss.
* Other parameters like `weight_decay` and `report_to`.

We also use `DataCollatorForLanguageModeling` which prepares batches for causal language modeling (ensuring inputs and labels are correctly formatted).

Finally, we initialize the `Trainer` with the base model, training arguments, processed dataset, data collator, and tokenizer.

In [54]:
# 8. Set Up Training Arguments and Initialize Trainer

# Configuration
drake_output_dir = "./drake-lyrics-model" # Directory for Drake model output
num_epochs = 15 # Final number of epochs after experimentation
batch_size_per_device = 4
learning_rate = 5e-5
processed_dataset_path = "./drake_lm_dataset" # Path to saved processed data

# Make sure model and tokenizer are loaded (from step 6)
if 'model' not in globals() or model is None or \
   'tokenizer' not in globals() or tokenizer is None:
    print("ERROR: Base model/tokenizer not loaded. Please run section 4 first.")
# Make sure processed dataset exists
elif not os.path.exists(processed_dataset_path):
    print(f"ERROR: Processed dataset not found at {processed_dataset_path}. Please run section 5 first.")
else:
    print(f"\nLoading processed dataset from {processed_dataset_path}...")
    try:
        # Load the processed dataset from disk
        lm_datasets = load_from_disk(processed_dataset_path)
        if 'train' not in lm_datasets:
             raise ValueError("Loaded dataset missing 'train' split.")
        train_dataset = lm_datasets['train'] # We only need the train split
        print("Processed dataset loaded successfully:")
        print(lm_datasets)

        # --- Define Training Arguments ---
        print("\nDefining Training Arguments...")
        training_args = TrainingArguments(
            output_dir=drake_output_dir,
            overwrite_output_dir=True,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size_per_device,
            save_strategy="epoch",
            logging_strategy="steps",
            logging_steps=50, # Log loss every 50 steps
            learning_rate=learning_rate,
            weight_decay=0.01,
            push_to_hub=False,
            report_to="none", # Disable external logging integrations
            # fp16=False, # Mixed precision - adjust if using GPU
        )
        print(f"Arguments defined for {num_epochs} epochs.")

        # --- Define Data Collator ---
        # For causal LM, mlm=False
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        print("Data collator defined.")

        # --- Initialize Trainer ---
        print("\nInitializing Trainer...")
        trainer = Trainer(
            model=model, # The base gpt2 model
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
            tokenizer=tokenizer, # Pass tokenizer for saving
        )
        print("Trainer initialized successfully!")
        print(f"\nTrainer is ready to fine-tune for {num_epochs} epochs.")

    except Exception as e:
        print(f"An error occurred during Trainer setup: {e}")
        trainer = None # Flag error


Loading processed dataset from ./drake_lm_dataset...
Processed dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1492
    })
})

Defining Training Arguments...
Arguments defined for 15 epochs.
Data collator defined.

Initializing Trainer...
Trainer initialized successfully!

Trainer is ready to fine-tune for 15 epochs.


<ipython-input-54-959095646d22>:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [55]:
# 9. Run Fine-Tuning

print("Starting Drake model fine-tuning...")
print(f"(This may take approx. 15 minutes for {num_epochs} epochs on TPU...)")

if 'trainer' in globals() and trainer is not None:
    try:
        # Start the training process
        train_result = trainer.train()

        # Save final model state and metrics
        trainer.save_model() # Saves to output_dir defined in TrainingArguments
        metrics = train_result.metrics
        metrics["train_samples"] = len(train_dataset) # Add sample count for context
        trainer.log_metrics("train", metrics)
        trainer.save_metrics("train", metrics)
        trainer.save_state()

        print("\n--- Training Finished ---")
        print("Final model state, metrics, and trainer state saved.")
        print("\nFinal Training Metrics:")
        print(metrics)

    except Exception as e:
        print(f"\nAn error occurred during training: {e}")
else:
    print("\nERROR: Trainer object not initialized. Cannot start training.")

Starting Drake model fine-tuning...
(This may take approx. 15 minutes for 15 epochs on TPU...)


Step,Training Loss
50,4.025100
100,3.876600
150,3.844100
200,3.722400
250,3.783500
300,3.701200
350,3.676800
400,3.504000
450,3.443400
500,3.442800


***** train metrics *****
  epoch                    =       15.0
  total_flos               =  1361527GF
  train_loss               =     2.4481
  train_runtime            = 0:14:21.76
  train_samples            =       1492
  train_samples_per_second =      25.97
  train_steps_per_second   =      6.492

--- Training Finished ---
Final model state, metrics, and trainer state saved.

Final Training Metrics:
{'train_runtime': 861.7684, 'train_samples_per_second': 25.97, 'train_steps_per_second': 6.492, 'total_flos': 1461928919040000.0, 'train_loss': 2.4480800720705744, 'epoch': 15.0, 'train_samples': 1492}


## Text Generation

With the model fine-tuned, we can now use it to generate new text. We load the best-performing checkpoint (from the 15-epoch training run).

We use the Hugging Face `pipeline` for text generation, which simplifies the process. Key parameters used during generation significantly impact the output quality:
* `prompt`: The initial text provided to the model to start generation.
* `max_length`: The maximum length of the generated sequence.
* `do_sample=True`: Enables sampling strategies instead of deterministic greedy decoding.
* `top_k` / `top_p`: Nucleus sampling parameters to control the pool of candidate tokens, balancing creativity and coherence.
* `temperature`: Controls the randomness of the sampling (lower values make it more focused, higher values more random).
* `repetition_penalty`: Discourages the model from repeating

In [57]:
# 10. Generate Text with the Fine-Tuned Model

from transformers import pipeline # Ensure pipeline is imported if running standalone

# --- Configuration ---
drake_model_output_dir = "./drake-lyrics-model" # Directory containing checkpoints
# Use a Drake-like prompt
prompt = "Yeah, okay"
# Generation parameters found to work best
gen_max_length = 80
gen_num_sequences = 3
gen_temperature = 0.7
gen_top_k = 50
gen_top_p = 0.95
gen_repetition_penalty = 1.2

# --- Find Latest Checkpoint ---
print(f"Searching for latest checkpoint in {drake_model_output_dir}...")
latest_checkpoint = None
if os.path.isdir(drake_model_output_dir):
    try:
        checkpoints = [
            d for d in os.listdir(drake_model_output_dir)
            if d.startswith("checkpoint-") and os.path.isdir(os.path.join(drake_model_output_dir, d))
        ]
        if checkpoints:
            checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
            latest_checkpoint = os.path.join(drake_model_output_dir, checkpoints[-1])
            print(f"Using latest checkpoint: {latest_checkpoint}")
        else:
            # Fallback if no checkpoint folders found (e.g., if only final model saved)
            if os.path.exists(os.path.join(drake_model_output_dir, "pytorch_model.bin")): # Basic check for saved model files
                 latest_checkpoint = drake_model_output_dir
                 print("No checkpoint folders found, attempting to load directly from output directory.")
            else:
                 print("No checkpoints or saved model found in output directory.")

    except Exception as e:
        print(f"Error finding latest checkpoint: {e}")
else:
     print(f"Output directory not found: {drake_model_output_dir}")


# --- Load Fine-tuned Model and Tokenizer ---
# Re-load the specific fine-tuned model version and its tokenizer
loaded_model = None
loaded_tokenizer = None
if latest_checkpoint and os.path.exists(latest_checkpoint):
    print("\nLoading fine-tuned Drake model and tokenizer from checkpoint...")
    try:
        # Load the fine-tuned model and tokenizer from the checkpoint
        loaded_tokenizer = AutoTokenizer.from_pretrained(latest_checkpoint)
        if loaded_tokenizer.pad_token is None: # Ensure pad token is set
             loaded_tokenizer.pad_token = loaded_tokenizer.eos_token

        loaded_model = AutoModelForCausalLM.from_pretrained(latest_checkpoint)
        print("Fine-tuned model and tokenizer loaded successfully!")

        # Determine device for pipeline (CPU recommended for pipeline on TPU runtime)
        if torch.cuda.is_available(): device = 0; print("Using GPU for generation.")
        elif 'COLAB_TPU_ADDR' in os.environ: device = -1; print("TPU detected, using CPU for pipeline generation.")
        else: device = -1; print("Using CPU for generation.")

    except Exception as e:
        print(f"Error loading fine-tuned model/tokenizer: {e}")
        loaded_model, loaded_tokenizer, device = None, None, -1
else:
    print("\nCould not find saved model checkpoint. Cannot generate text.")
    loaded_model, loaded_tokenizer, device = None, None, -1


# --- Generate Text using Pipeline ---
if loaded_model and loaded_tokenizer:
    print("\nInitializing text generation pipeline...")
    try:
        # Create the text generation pipeline
        generator = pipeline(
            'text-generation',
            model=loaded_model,
            tokenizer=loaded_tokenizer,
            device=device # Use CPU or GPU
        )
        print("Pipeline initialized.")

        print("\nGenerating text samples...")
        # Generate text with specified parameters
        generated_texts = generator(
            prompt,
            max_length=gen_max_length,
            num_return_sequences=gen_num_sequences,
            do_sample=True,
            top_k=gen_top_k,
            top_p=gen_top_p,
            temperature=gen_temperature,
            repetition_penalty=gen_repetition_penalty
        )

        print(f"\n--- Generated Text Examples (Prompt: '{prompt}') ---")
        for i, text in enumerate(generated_texts):
            print(f"\n--- Generation {i+1} ---")
            # Print only the generated text part
            print(text['generated_text'])

    except Exception as e:
        print(f"\nAn error occurred during text generation: {e}")

else:
    print("\nSkipping text generation due to errors loading the fine-tuned model.")

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Searching for latest checkpoint in ./drake-lyrics-model...
Using latest checkpoint: ./drake-lyrics-model/checkpoint-5685

Loading fine-tuned Drake model and tokenizer from checkpoint...
Fine-tuned model and tokenizer loaded successfully!
Using CPU for generation.

Initializing text generation pipeline...
Pipeline initialized.

Generating text samples...

--- Generated Text Examples (Prompt: 'Yeah, okay') ---

--- Generation 1 ---
Yeah, okayGuess that's just the motionOh-ohI guess that’s just how I feelIf I'm working then you should knowThat your phone is blowing upAnd my friend calling askin' me to stopIt seem like he or she don't know what they areDo they really wanna be friends?Are we too late for all this drama?"These days men are busy

--- Generation 2 ---
Yeah, okay)I'm on Air One (Okay), issa nine karatIssa Pyrex and a Range RoverBlack Benz coupe with tinted windowsAnd my nigga 'bout to sell you shit like Bausch & LombBut I play myself in the stereoAnd tell you not to ever switch

## Qualitative Analysis

* **Coherence:**
    * The generated lines do not make a lot of sense, but do seem to be attempting to mimic Drake's style.
    * The progression of ideas is quite random, showing the model does really understand the context of what it just wrote
    * Initially the model was run with 3 epochs and no repetition penalty and the output was terrible. The model would outpot repeated phrases and have no coherence at all. Adding a penalty for repetitiveness seemd to help fix this issues.

* **Relevance (Style & Content):**
    * The model does sound a little bit lke Drake, using slang and expressing the bold "attitude" in Drake's music.
    * It doesn't really read like rap lyrics, but I kind of expected this using a small model like GPT-2
    * The themes of the generated text seem to be somewhat accurate, capturing the romantic pop style theme of many of Drake's sone.

* **Creativity & Repetition:**
    * It seems to be attempting to generate novel lines, and it also seems this affects the coherence of the lyrics.

* **Overall Assessment & Limitations:**
    * In general, I don't think the fine-tuning was any good at generating the type of lyrics that Drake could.
    * I think the main limitations is just the overall intelligence of the GPT-2 model and it's ability to understand context.
    * I originally tried this with my ownd dataset by manually copying and pasting lyrics from Jack Harlow songs on Genius. And the output was terrible, very repetitive and little to no coherence. Having a larger dataset seems to be absolutely necessary. Maybe if I had an even larger dataset than the one use in this project, I could see even better results.


## Replication Instructions

To replicate the fine-tuning process and results for the Drake lyrics generation model:

1.  **Environment:**
    * Use Google Colab ([colab.research.google.com](https://colab.research.google.com/)).
    * Set the runtime type to **TPU** (`Runtime` -> `Change runtime type` -> `Hardware accelerator: TPU`). Using a GPU (like T4 or V100) should also work but may require adjusting batch size or enabling `fp16` in `TrainingArguments` depending on memory; TPU is recommended as used here.

2.  **Libraries:**
    * Run the first code cell (cell #1) in the notebook to install the required libraries:
        ```bash
        !pip install transformers datasets torch accelerate evaluate pandas -q
        ```

3.  **Data:**
    * Download the Drake lyrics dataset (`drake_data.csv`) from Kaggle: [https://www.kaggle.com/datasets/juicobowley/drake-lyrics](https://www.kaggle.com/datasets/juicobowley/drake-lyrics)
    * Upload the `drake_data.csv` file to the root directory of your Google Colab session environment (use the "Files" tab on the left sidebar).

4.  **Execution:**
    * Run all the code cells in this notebook sequentially from top to bottom (using `Runtime` -> `Run all` or executing each cell individually).

5.  **Key Parameters & Configuration:**
    * **Base Model:** `gpt2`
    * **Dataset:** Drake Lyrics from Kaggle (as linked above).
    * **Training Epochs:** 15
    * **Batch Size (per device):** 4
    * **Learning Rate:** 5e-5
    * **Block Size:** 128
    * **Generation Settings (Cell #10):** Includes `temperature=0.7`, `repetition_penalty=1.2`.

6.  **Expected Outcome:**
    * The script will clean the data, prepare it, and fine-tune the GPT-2 model.
    * Training takes approximately 15 minutes on a Colab TPU.
    * A fine-tuned model will be saved in the `./drake-lyrics-model` directory within the Colab environment, with checkpoints saved after each epoch.
    * Running the final text generation cell (cell #10) should produce samples of Drake-style lyrics similar to those shown in the notebook's output, demonstrating successful fine-tuning. The exact output will vary slightly due to the sampling process.